In [1]:
import getpass
import os
import pprint


def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)


_set_if_undefined("ANTHROPIC_API_KEY")
_set_if_undefined("TAVILY_API_KEY")

In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

search = TavilySearchAPIWrapper()
tavily_tool = TavilySearchResults(api_wrapper=search, max_results=5)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18304\3459882989.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(api_wrapper=search, max_results=5)


In [3]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="anthropic",
    base_url="https://api.deepseek.com/anthropic", # 必须补充此核心参数
    temperature=0)

ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (D:\project\py\finance_agent\.venv\Lib\site-packages\langchain_core\exceptions.py)

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.output_parsers.openai_tools import PydanticToolsParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pydantic import ValidationError

from pydantic import BaseModel, Field


class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous")


class AnswerQuestion(BaseModel):
    """Answer the question. Provide an answer, reflection, and then follow up with search queries to improve the answer."""

    answer: str = Field(description="~250 word detailed answer to the question.")
    reflection: Reflection = Field(description="Your reflection on the initial answer.")
    search_queries: list[str] = Field(
        description="1-3 search queries for researching improvements to address the critique of your current answer."
    )


class ResponderWithRetries:
    def __init__(self, runnable, validator):
        self.runnable = runnable
        self.validator = validator

    def respond(self, state: dict):
        response = []
        for attempt in range(3):
            response = self.runnable.invoke(
                {"messages": state["messages"]}, {"tags": [f"attempt:{attempt}"]}
            )
            try:
                self.validator.invoke(response)
                return {"messages": response}
            except ValidationError as e:
                state = {
                    **state,
                    "messages": state["messages"] + [
                        response,
                        ToolMessage(
                            content=f"{repr(e)}\n\nPay close attention to the function schema.\n\n"
                            + self.validator.schema_json()
                            + " Respond by fixing all validation errors.",
                            tool_call_id=response.tool_calls[0]["id"],
                        ),
                    ],
                }
        return {"messages": response}

import datetime

actor_prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are expert researcher.
Current time: {time}

1. {first_instruction}
2. Reflect and critique your answer. Be severe to maximize improvement.
3. Recommend search queries to research information and improve your answer.""",
        ),
        MessagesPlaceholder(variable_name="messages"),
        (
            "user",
            "\n\n<reminder>Reflect on the user's original question and the"
            " actions taken thus far. Respond using the {function_name} function.</reminder>",
        ),
    ]
).partial(
    time=lambda: datetime.datetime.now().isoformat(),
)
initial_answer_chain = actor_prompt_template.partial(
    first_instruction="Provide a detailed ~250 word answer.",
    function_name=AnswerQuestion.__name__,
) | llm.bind_tools(tools=[AnswerQuestion])
validator = PydanticToolsParser(tools=[AnswerQuestion])

first_responder = ResponderWithRetries(
    runnable=initial_answer_chain, validator=validator
)

example_question = "Why is reflection useful in AI?"
initial = first_responder.respond(
    {"messages": [HumanMessage(content=example_question)]}
)


revise_instructions = """Revise your previous answer using the new information.
    - You should use the previous critique to add important information to your answer.
        - You MUST include numerical citations in your revised answer to ensure it can be verified.
        - Add a "References" section to the bottom of your answer (which does not count towards the word limit). In form of:
            - [1] https://example.com
            - [2] https://example.com
    - You should use the previous critique to remove superfluous information from your answer and make SURE it is not more than 250 words.
"""


# Extend the initial answer schema to include references.
# Forcing citation in the model encourages grounded responses
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question. Provide an answer, reflection,

    cite your reflection with references, and finally
    add search queries to improve the answer."""

    references: list[str] = Field(
        description="Citations motivating your updated answer."
    )


revision_chain = actor_prompt_template.partial(
    first_instruction=revise_instructions,
    function_name=ReviseAnswer.__name__,
) | llm.bind_tools(tools=[ReviseAnswer])
revision_validator = PydanticToolsParser(tools=[ReviseAnswer])

revisor = ResponderWithRetries(runnable=revision_chain, validator=revision_validator)

In [ ]:
import json

revised = revisor.respond(
    {
        "messages": [
            HumanMessage(content=example_question),
            initial["messages"],
            ToolMessage(
                tool_call_id=initial["messages"].tool_calls[0]["id"],
                content=json.dumps(
                    tavily_tool.invoke(
                        {
                            "query": initial["messages"].tool_calls[0]["args"][
                                "search_queries"
                            ][0]
                        }
                    )
                ),
            ),
        ]
    }
)
revised["messages"]

In [ ]:
from IPython.display import Markdown, display
import json
from IPython.display import Markdown, display

def show_message(msg):
    t = msg.type   # 'human' | 'ai' | 'tool'

    # ── HumanMessage ──
    if t == "human":
        display(Markdown(f"### 👤 Human\n\n{msg.content}"))
        return

    # ── ToolMessage(搜索结果) ──
    if t == "tool":
        import json
        try:
            data = json.loads(msg.content)
            preview = json.dumps(data, ensure_ascii=False, indent=2)[:800]
        except Exception:
            preview = str(msg.content)[:800]
        display(Markdown(f"### 🌐 Tool Result\n\n```json\n{preview}\n```"))
        return

    # ── AIMessage ──
    parts = []
    # content 可能是字符串(OpenAI)或块列表(Claude 的 thinking)
    if isinstance(msg.content, list):
        for b in msg.content:
            if isinstance(b, dict) and b.get("type") == "thinking":
                parts.append("### 🧠 Thinking\n> " + b["thinking"].replace("\n", "\n> "))
    elif isinstance(msg.content, str) and msg.content.strip():
        parts.append(msg.content)

    # tool_calls 用 getattr 兜底，没有就空列表
    for tc in getattr(msg, "tool_calls", []) or []:
        args = tc["args"]
        parts.append(f"### 🔧 {tc['name']}")
        if "answer" in args:
            parts.append(f"**📝 Answer**\n\n{args['answer']}")
        if "reflection" in args:
            r = args["reflection"]
            parts.append(f"**🔍 Reflection**\n- Missing: {r.get('missing','')}\n- Superfluous: {r.get('superfluous','')}")
        if "search_queries" in args:
            qs = "\n".join(f"{i+1}. `{q}`" for i, q in enumerate(args["search_queries"]))
            parts.append(f"**🔎 Search Queries**\n\n{qs}")
        if "references" in args:
            parts.append("**📚 References**\n\n" + "\n".join(f"- {u}" for u in args["references"]))

    display(Markdown("\n\n---\n\n".join(parts) if parts else f"*(空 AI 消息)*"))
show_message(initial["messages"])
show_message(revised["messages"])

In [ ]:
from langchain_core.tools import StructuredTool

from langgraph.prebuilt import ToolNode


def run_queries(search_queries: list[str], **kwargs):
    """Run the generated queries."""
    return tavily_tool.batch([{"query": query} for query in search_queries])


tool_node = ToolNode(
    [
        StructuredTool.from_function(run_queries, name=AnswerQuestion.__name__),
        StructuredTool.from_function(run_queries, name=ReviseAnswer.__name__),
    ]
)

In [ ]:
from typing import Literal

from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict


class State(TypedDict):
    messages: Annotated[list, add_messages]


MAX_ITERATIONS = 5
builder = StateGraph(State)
builder.add_node("draft", first_responder.respond)


builder.add_node("execute_tools", tool_node)
builder.add_node("revise", revisor.respond)
# draft -> execute_tools
builder.add_edge("draft", "execute_tools")
# execute_tools -> revise
builder.add_edge("execute_tools", "revise")

# Define looping logic:


def _get_num_iterations(state: list):
    i = 0
    for m in state[::-1]:
        if m.type not in {"tool", "ai"}:
            break
        i += 1
    return i


def event_loop(state: list):
    # in our case, we'll just stop after N plans
    num_iterations = _get_num_iterations(state["messages"])
    if num_iterations > MAX_ITERATIONS:
        return END
    return "execute_tools"


# revise -> execute_tools OR end
builder.add_conditional_edges("revise", event_loop, ["execute_tools", END])
builder.add_edge(START, "draft")
graph = builder.compile()




In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
events = graph.stream(
    {"messages": [("user", "How should we handle the climate crisis?")]},
    stream_mode="values",
)


In [ ]:
for i, step in enumerate(events):
    show_message(step["messages"][-1])